In [1]:
import numpy as np

from scipy.stats import norm
from scipy.spatial import cKDTree

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    ConstantKernel,
    Matern,
    WhiteKernel
)

# ============================================================
# FUNCTION 2 - WEEK 11 BAYESIAN OPTIMISATION
# Run from inside week11/
# ============================================================
#
# Strategy:
# - Refit ARD Matern GP including Week 10.
# - Check Week 10 calibration.
# - Search around the actual incumbent.
# - Retain moderate wider/global candidates for diagnostics.
# - Compare EI, posterior mean and UCB.
# ============================================================


# ------------------------------------------------------------
# 1. Load cumulative Week 11 data
# ------------------------------------------------------------

X = np.load("function2/initial_inputs.npy")
Y = np.load("function2/initial_outputs.npy").reshape(-1)

best_idx = np.argmax(Y)
best_x = X[best_idx]
best_y = Y[best_idx]

print("================================")
print("DATA")
print("================================")

print("X shape:", X.shape)
print("Y shape:", Y.shape)

print("\nCurrent best:")
print(best_x, "->", best_y)

print("\nY range:")
print("min =", Y.min())
print("max =", Y.max())
print("std =", Y.std())


# ------------------------------------------------------------
# 2. Week 10 calibration check
# ------------------------------------------------------------
#
# Week 10 selected:
# [0.702316, 0.995076]
#
# Use the Week 10 predicted mean/std from the previous notebook.
# These values were approximately:
#
# mean ≈ 0.612648
# std  ≈ 0.060858
#
# Actual:
# 0.6682000915555967
# ------------------------------------------------------------

week10_pred_mean = 0.612648
week10_pred_std = 0.060858
week10_actual = 0.6682000915555967

week10_error = (
    week10_actual
    - week10_pred_mean
)

week10_z_error = (
    week10_error
    / week10_pred_std
)

print("\n================================")
print("WEEK 10 CALIBRATION CHECK")
print("================================")

print("Predicted mean:", week10_pred_mean)
print("Predicted std :", week10_pred_std)
print("Actual        :", week10_actual)

print("\nPrediction error:")
print(week10_error)

print("\nError / predicted std:")
print(week10_z_error)


# ------------------------------------------------------------
# 3. Fit ARD Matern GP
# ------------------------------------------------------------

kernel = (
    ConstantKernel(
        1.0,
        constant_value_bounds=(1e-3, 1e3)
    )
    *
    Matern(
        length_scale=np.ones(2) * 0.2,
        length_scale_bounds=(0.01, 2.0),
        nu=2.5
    )
    +
    WhiteKernel(
        noise_level=1e-5,
        noise_level_bounds=(1e-8, 1e-1)
    )
)

gp = GaussianProcessRegressor(
    kernel=kernel,
    normalize_y=True,
    n_restarts_optimizer=30,
    random_state=42
)

gp.fit(X, Y)

print("\n================================")
print("GP FIT")
print("================================")

print("\nFitted kernel:")
print(gp.kernel_)

lengthscales = gp.kernel_.k1.k2.length_scale

inverse_ls = 1.0 / lengthscales

relative_sensitivity = (
    inverse_ls
    / inverse_ls.sum()
)

print("\nARD lengthscales:")
print(lengthscales)

print(
    "\nNormalised inverse-lengthscale sensitivity:"
)
print(relative_sensitivity)


# ------------------------------------------------------------
# 4. Expected Improvement
# ------------------------------------------------------------

def expected_improvement(
    mu,
    sigma,
    best_y,
    xi=0.0
):

    improvement = (
        mu - best_y - xi
    )

    valid = sigma > 1e-12

    Z = np.zeros_like(mu)

    Z[valid] = (
        improvement[valid]
        / sigma[valid]
    )

    EI = np.zeros_like(mu)

    EI[valid] = (
        improvement[valid]
        * norm.cdf(Z[valid])
        +
        sigma[valid]
        * norm.pdf(Z[valid])
    )

    return EI


# ------------------------------------------------------------
# 5. Candidate generation
# ------------------------------------------------------------

rng = np.random.default_rng(42)

local_scale = np.clip(
    0.20 * lengthscales,
    0.015,
    0.08
)

wide_scale = np.clip(
    0.40 * lengthscales,
    0.04,
    0.15
)

print("\n================================")
print("CANDIDATE SCALES")
print("================================")

print("Local widths:")
print(local_scale)

print("\nWide widths:")
print(wide_scale)


local_candidates = (
    best_x
    + rng.normal(
        0,
        local_scale,
        size=(140000, 2)
    )
)

wide_candidates = (
    best_x
    + rng.normal(
        0,
        wide_scale,
        size=(100000, 2)
    )
)

global_candidates = rng.uniform(
    0,
    1,
    size=(80000, 2)
)

local_candidates = np.clip(
    local_candidates,
    0,
    1
)

wide_candidates = np.clip(
    wide_candidates,
    0,
    1
)


# ------------------------------------------------------------
# 6. Explicit x2 = 1 boundary coverage
# ------------------------------------------------------------

x1_line = np.linspace(
    0.55,
    0.85,
    12001
)

boundary_line = np.column_stack([
    x1_line,
    np.ones_like(x1_line)
])


candidates = np.vstack([
    local_candidates,
    wide_candidates,
    global_candidates,
    boundary_line
])


# ------------------------------------------------------------
# 7. Remove near-duplicates
# ------------------------------------------------------------

tree = cKDTree(X)

distance, _ = tree.query(
    candidates,
    k=1
)

candidates = candidates[
    distance > 0.01
]

print("\nCandidates after duplicate filtering:")
print(len(candidates))


# ------------------------------------------------------------
# 8. Predictions
# ------------------------------------------------------------

mu, sigma = gp.predict(
    candidates,
    return_std=True
)


# ------------------------------------------------------------
# 9. Primary EI
# ------------------------------------------------------------

EI = expected_improvement(
    mu,
    sigma,
    best_y,
    xi=0.0
)

ei_idx = np.argmax(EI)

print("\n================================")
print("PRIMARY EI")
print("================================")

print("candidate =", candidates[ei_idx])
print("mean =", mu[ei_idx])
print("std =", sigma[ei_idx])
print("EI =", EI[ei_idx])


# ------------------------------------------------------------
# 10. Highest predicted mean
# ------------------------------------------------------------

mean_idx = np.argmax(mu)

print("\n================================")
print("HIGHEST PREDICTED MEAN")
print("================================")

print("candidate =", candidates[mean_idx])
print("mean =", mu[mean_idx])
print("std =", sigma[mean_idx])


# ------------------------------------------------------------
# 11. UCB diagnostics
# ------------------------------------------------------------

print("\n================================")
print("UCB DIAGNOSTICS")
print("================================\n")

for beta in [
    0.05,
    0.10,
    0.25,
    0.50,
    1.00
]:

    UCB = (
        mu
        + beta * sigma
    )

    idx = np.argmax(UCB)

    print(
        f"beta={beta}",
        "\n candidate =", candidates[idx],
        "\n mean =", round(mu[idx], 6),
        "\n std =", round(sigma[idx], 6),
        "\n UCB =", round(UCB[idx], 6),
        "\n"
    )


# ------------------------------------------------------------
# 12. Distance from incumbent
# ------------------------------------------------------------

print("\n================================")
print("DISTANCE FROM CURRENT BEST")
print("================================")

print(
    "EI:",
    np.linalg.norm(
        candidates[ei_idx]
        - best_x
    )
)

print(
    "Highest mean:",
    np.linalg.norm(
        candidates[mean_idx]
        - best_x
    )
)

for beta in [
    0.05,
    0.10,
    0.25,
    0.50,
    1.00
]:

    UCB = mu + beta * sigma
    idx = np.argmax(UCB)

    print(
        f"UCB beta={beta}:",
        np.linalg.norm(
            candidates[idx]
            - best_x
        )
    )


# ------------------------------------------------------------
# 13. Domain-boundary diagnostic
# ------------------------------------------------------------

def domain_boundary_status(
    x,
    tol=0.01
):

    status = []

    for j in range(len(x)):

        if x[j] <= tol:
            status.append(
                f"x{j+1}~0"
            )

        elif x[j] >= 1.0 - tol:
            status.append(
                f"x{j+1}~1"
            )

    if not status:
        return "interior"

    return ", ".join(status)


print("\n================================")
print("DOMAIN BOUNDARY CHECK")
print("================================")

print(
    "EI:",
    domain_boundary_status(
        candidates[ei_idx]
    )
)

print(
    "Highest mean:",
    domain_boundary_status(
        candidates[mean_idx]
    )
)

for beta in [
    0.05,
    0.10,
    0.25,
    0.50,
    1.00
]:

    UCB = mu + beta * sigma
    idx = np.argmax(UCB)

    print(
        f"UCB beta={beta}:",
        domain_boundary_status(
            candidates[idx]
        )
    )

DATA
X shape: (20, 2)
Y shape: (20,)

Current best:
[0.711177 1.      ] -> 0.679662733033218

Y range:
min = -0.06562362443733738
max = 0.679662733033218
std = 0.2513238129663095

WEEK 10 CALIBRATION CHECK
Predicted mean: 0.612648
Predicted std : 0.060858
Actual        : 0.6682000915555967

Prediction error:
0.055552091555596705

Error / predicted std:
0.9128149389660637


/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(



GP FIT

Fitted kernel:
1.05**2 * Matern(length_scale=[0.0585, 2], nu=2.5) + WhiteKernel(noise_level=0.0423)

ARD lengthscales:
[0.0585037 2.       ]

Normalised inverse-lengthscale sensitivity:
[0.9715795 0.0284205]

CANDIDATE SCALES
Local widths:
[0.015 0.08 ]

Wide widths:
[0.04 0.15]

Candidates after duplicate filtering:
243975

PRIMARY EI
candidate = [9.71627967e-01 4.33842900e-04]
mean = 0.3848986784115077
std = 0.26036291761158725
EI = 0.016760079032772407

HIGHEST PREDICTED MEAN
candidate = [0.7023142  0.97204994]
mean = 0.6261256450514625
std = 0.05715275606197795

UCB DIAGNOSTICS

beta=0.05 
 candidate = [0.70225311 0.97207089] 
 mean = 0.626125 
 std = 0.057192 
 UCB = 0.628984 

beta=0.1 
 candidate = [0.70225311 0.97207089] 
 mean = 0.626125 
 std = 0.057192 
 UCB = 0.631844 

beta=0.25 
 candidate = [0.70184169 0.97212513] 
 mean = 0.626086 
 std = 0.057465 
 UCB = 0.640452 

beta=0.5 
 candidate = [0.7011841  0.97223827] 
 mean = 0.625909 
 std = 0.057946 
 UCB = 0.6548

In [2]:
# ============================================================
# FINAL FUNCTION 2 - WEEK 11 SELECTION
# ============================================================

final_idx = np.argmax(mu)

week11_candidate = candidates[final_idx]

print("Week 11 Function 2 candidate:")
print(week11_candidate)

print("\nPredicted mean:")
print(mu[final_idx])

print("\nPredicted std:")
print(sigma[final_idx])

print("\nDistance from current best:")
print(
    np.linalg.norm(
        week11_candidate - best_x
    )
)

portal = "-".join(
    f"{x:.6f}"
    for x in week11_candidate
)

print("\nPortal format:")
print(portal)

Week 11 Function 2 candidate:
[0.7023142  0.97204994]

Predicted mean:
0.6261256450514625

Predicted std:
0.05715275606197795

Distance from current best:
0.029321583862178507

Portal format:
0.702314-0.972050
